# Analysis of experiment #1: DPCP heuristics

## Read data

In [1]:
# Read csv

import pandas as pd

df = pd.read_csv("stats3.csv")

# Some renames to improve readability
df["solver"] = df["solver"].str.replace("v2", "deg")
df["solver"] = df["solver"].str.replace("v3", "edg")

# For the unsolved instances,
# --> write NaN in ub, bestTime, bestIter
df.loc[df.state != "FEASIBLE", "value"] = float("nan")
df.loc[df.state != "FEASIBLE", "bestTime"] = float("nan")
df.loc[df.state != "FEASIBLE", "bestIter"] = float("nan")

# Rename some columns
df = df.rename(columns={'state': 'solved'})

# Add column with edge probability
df["p"] = df.instance.apply(lambda s: float(s.split("_")[2][1:]))

# Rename heuristic names
df["solver"] = df["solver"].replace({
    "heur_greedy2s_deg": "Greedy 2-step (DEG)",
    "heur_greedy2s_edg": "Greedy 2-step (EDG)",
    "heur_semigreedy2s_deg": "Semigreedy 2-step (DEG)",
    "heur_semigreedy2s_edg": "Semigreedy 2-step (EDG)",
    "heur_greedy1s": "Greedy 1-step",
})

df

,instance,solver,run,nvertices,nedges,n,m,solved,value,totalTime,totalIters,bestTime,bestIter,p
0,r_n170_p0.75_nA17_nB34_i1,Greedy 2-step (DEG),0,170,10781,17,34,FEASIBLE,6.0,0.003148,1,0.003148,0.0,0.75
1,r_n170_p0.75_nA17_nB17_i1,Greedy 2-step (DEG),0,170,10787,17,17,FEASIBLE,7.0,0.002770,1,0.002769,0.0,0.75
2,r_n130_p0.25_nA26_nB26_i1,Greedy 2-step (DEG),0,130,2071,26,26,FEASIBLE,4.0,0.000916,1,0.000915,0.0,0.25
3,r_n150_p0.5_nA15_nB30_i2,Greedy 2-step (DEG),0,150,5529,15,30,FEASIBLE,4.0,0.001662,1,0.001661,0.0,0.50
4,r_n150_p0.5_nA15_nB15_i4,Greedy 2-step (DEG),0,150,5682,15,15,FEASIBLE,5.0,0.001686,1,0.001686,0.0,0.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,r_n130_p0.25_nA26_nB26_i2,Greedy 1-step,0,130,2100,26,26,FEASIBLE,3.0,0.002944,1,0.002944,0.0,0.25
246,r_n170_p0.75_nA34_nB17_i1,Greedy 1-step,0,170,10786,34,17,UNKNOWN,NaN,0.009998,1,NaN,NaN,0.75
247,r_n150_p0.5_nA30_nB30_i0,Greedy 1-step,0,150,5548,30,30,FEASIBLE,7.0,0.006714,1,0.006714,0.0,0.50
248,r_n150_p0.5_nA30_nB15_i1,Greedy 1-step,0,150,5541,30,15,UNKNOWN,NaN,0.005603,1,NaN,NaN,0.50


## Best 

In [2]:
# Pivot dataframe
df2 = df.pivot(index="instance", columns="solver", values=["value","totalTime"])
n_heurs = int(len(df2.columns)/2)

In [3]:
# Comparison function
def get_winner_heur(row):
    best_heur = None
    min_value = float("inf")
    min_time = float("inf")
    for i in range(n_heurs):
        h = df2.columns[i][1] # get heuristic's name
        j = ("totalTime", h) # get column's name of the bestTime for h
        if (row.iloc[i] < min_value) or (row.iloc[i] == min_value and row[j] < min_time):
            best_heur = h
            min_value = row.iloc[i]
            min_time = row[j]
    return best_heur

def get_winner_value(row):
    min_value = float("inf")
    for i in range(n_heurs):
        if (row.iloc[i] < min_value):
            min_value = row.iloc[i]
    return min_value

In [4]:
df2["winnerValue"] = df2.apply(get_winner_value, axis=1)
df2["winner"] = df2.apply(get_winner_heur, axis=1)

In [5]:
df["winner"] = df.apply(lambda row: row.solver == df2.loc[row.instance].winner, axis=1)
df["tie"] = df.apply(lambda row: (row.value == df2.loc[row.instance].winnerValue), axis=1)

## Summary

In [6]:
df3 = df.groupby(["solver"]).agg(
    {
        "solved": [("count", lambda x: (x == "FEASIBLE").sum())],
        "value": [("sum", "sum"), ("avg", "mean")],
        "winner": [("%", lambda x: (x == True).sum() / len(x) * 100)],
        "tie": [("%", lambda x: (x == True).sum() / len(x) * 100)],
        "totalTime": [("avg", "mean")],
    }
).reset_index()
df3.columns = ["Heuristic", "#Solved", "Σ Value", "Avg Value", "Best (%)", "≥ Best (%)", "Time (s)"]
df3 = df3.round({
    "Avg Value": 2,
    "Time (s)": 3,
})
df3["Σ Value"] = df3["Σ Value"].astype(int)
df3["Best (%)"] = df3["Best (%)"].round(0).astype(int)
df3["≥ Best (%)"] = df3["≥ Best (%)"].round(0).astype(int)
df3

,Heuristic,#Solved,Σ Value,Avg Value,Best (%),≥ Best (%),Time (s)
0,Greedy 1-step,36,197,5.47,14,20,0.007
1,Greedy 2-step (DEG),46,303,6.59,0,0,0.002
2,Greedy 2-step (EDG),46,306,6.65,6,6,0.002
3,Semigreedy 2-step (DEG),50,269,5.38,24,86,76.239
4,Semigreedy 2-step (EDG),50,272,5.44,56,82,59.731


In [7]:
df3 = df.groupby(["p","solver"]).agg(
    {
        "solved": [("count", lambda x: (x == "FEASIBLE").sum())],
        "value": [("sum", "sum"), ("avg", "mean")],
        "winner": [("%", lambda x: (x == True).sum() / len(x) * 100)],
        "tie": [("%", lambda x: (x == True).sum() / len(x) * 100)],
        "totalTime": [("avg", "mean")],
    }
).reset_index()
df3.columns = ["p", "Heuristic", "#Solved", "Σ Value", "Avg Value", "Best (%)", "≥ Best (%)", "Time (s)"]
df3 = df3.round({
    "Avg Value": 2,
    "Time (s)": 3,
})
df3["Σ Value"] = df3["Σ Value"].astype(int)
df3["Best (%)"] = df3["Best (%)"].round(0).astype(int)
df3["≥ Best (%)"] = df3["≥ Best (%)"].round(0).astype(int)
df3

,p,Heuristic,#Solved,Σ Value,Avg Value,Best (%),≥ Best (%),Time (s)
0,0.25,Greedy 1-step,7,26,3.71,20,30,0.003
1,0.25,Greedy 2-step (DEG),10,45,4.50,0,0,0.001
2,0.25,Greedy 2-step (EDG),10,43,4.30,10,10,0.001
3,0.25,Semigreedy 2-step (DEG),10,31,3.10,0,90,29.996
4,0.25,Semigreedy 2-step (EDG),10,30,3.00,70,100,22.390
5,0.50,Greedy 1-step,15,68,4.53,10,15,0.005
6,0.50,Greedy 2-step (DEG),20,120,6.00,0,0,0.002
7,0.50,Greedy 2-step (EDG),20,121,6.05,5,5,0.001
8,0.50,Semigreedy 2-step (DEG),20,89,4.45,10,75,66.772
9,0.50,Semigreedy 2-step (EDG),20,86,4.30,75,90,49.320
